## Overview
The following notebook seeks to reproduce the ITN usage and access statistics from the [2024-2025 Uganda Malaria Indicator Survey](https://dhsprogram.com/publications/publication-MIS45-MIS-Final-Reports.cfm). 

### Variables

`hhid`: Household ID

`hv001`: cluster

`hvidx`: Individual line number (individual-level)

`hv005`: Weight (household-level)

`hv024`: Region (household-level)

`hv103`: Stayed last night (y/n/missing) (person-level)

`hmlidx`: Net number (net-level)

`hml1`: Number of nets in household (household-level)

`hml10`: Net is ITN (net-level)

`hml12`: Type of net person slept under last night (person-level)

In [52]:
import zipfile
import numpy as np
import pandas as pd
from pathlib import Path

In [53]:
# Read in HMR
HMRpath = "../data/raw/UGPR91DT.zip"

with zipfile.ZipFile(HMRpath) as zip_ref:
    zip_ref.extractall("../data/raw")
    print("Extracted files:")
    for file in zip_ref.namelist():
        print(file)

rawHMR= pd.read_stata("../data/raw/UGPR91FL.dta")

Extracted files:
UGPR91FL.frq
UGPR91FL.frw
UGPR91FL.map
UGPR91FL.dta
UGPR91FL.DO
UGPR91FL.DCT


In [54]:
cols = ['hhid', 'hv001', 'hvidx', 'hv005', 'hv024', 'hv103', 'hmlidx', 
        'hml10', 'hml12', 'hml1', 'hml1a']
hmr = rawHMR[cols]

# Filter to de facto members
hmr_df = hmr[hmr['hv103'] == "yes"].copy()
records_dropped = len(hmr) - len(hmr_df)
print("Records dropped:", records_dropped)

Records dropped: 1352


## ITN Use by Region
The sample design of the 2024-25 Uganda MIS uses separate sampling frames for refugee settlements and non-refugee areas. Therefore the weights are calibrated to make the non-refugee sample representative of the non-refugee Ugandan population. Since the refugee settlements have different weights, they are dropped from the indicator calculations.


**Numerator:** all of de facto household members in region that slept under an ITN last night
**Denominator:** all of de facto household members in the region

In [55]:
# Create "slept under ITN" column
ITN_pos = ['only treated (itn) nets', 'both treated (itn) and untreated nets']
hmr_df['slept_ITN'] = hmr_df['hml12'].isin(ITN_pos)

# Create numerator column (slept_ITN x weight)
hmr_df['use_numerator'] = hmr_df.hv005 * hmr_df.slept_ITN

# Group by region
region_df_use = hmr_df[['hv024', 'hv005', 'slept_ITN', 'use_numerator']].groupby('hv024', observed=True).sum().reset_index()

# Drop refugee settlements
region_df_use = region_df_use[region_df_use['hv005'] != 0]

# Divide
region_df_use['use_rate'] = region_df_use["use_numerator"]/region_df_use["hv005"]

In [56]:
region_df_use

,hv024,hv005,slept_ITN,use_numerator,use_rate
0,kampala,1014956700,1345,560622900,0.552361
1,south buganda,7169112974,1901,4244450342,0.592047
2,north buganda,5120003785,2104,3361941885,0.656629
3,busoga,3797009509,2042,2461589335,0.648297
4,bukedi,2585607226,2002,1679731306,0.649647
5,elgon,2059554498,2073,1389564440,0.674692
6,teso,2367811911,2384,1845445847,0.779389
7,lango,2437319984,2207,1872548273,0.768282
8,acholi,1953488373,1696,1138711746,0.582912
9,karamoja,1424291664,1630,781591974,0.548758


## Access to ITNs
Because the Household Member Recode is at the individual-level, it does not include relevant information for nets a household owns but does not use. Therefore, to access characteristics of nets the household owns even if they don't use them, the Household Recode is necessary. This dataset contains characteristics for all individual nets (`hmlidx_a`, `hmlidx_b`, ..., `hmlidx_n`).

Access is defined as percentage of the household population that could sleep under an ITN if each ITN in the household were used by up to two people. 

**Numerator:** Sum of min(ITNs * 2, household size) across all households in a region

**Denominator:** Sum of household size across all households in a region

In [57]:
# Read in Household Recode data
HRpath = "../data/raw/UGHR91DT.zip"

with zipfile.ZipFile(HRpath) as zip_ref:
    zip_ref.extractall("../data/raw")
    print("Extracted files:")
    for file in zip_ref.namelist():
        print(file)

rawHR= pd.read_stata("../data/raw/UGHR91FL.dta")


Extracted files:
UGHR91FL.frq
UGHR91FL.frw
UGHR91FL.map
UGHR91FL.dta
UGHR91FL.DO
UGHR91FL.DCT


In [58]:
# Build total ITN df
hr_cols = ['hhid']
hml10 = list(rawHR.filter(like='hml10'))
hr_cols.extend(hml10)

hr_df = rawHR[hr_cols].copy()
hr_df['total_itns'] = (hr_df[hml10] == 'yes').sum(axis=1)

total_itns_df = hr_df[['hhid', 'total_itns']]

In [59]:
itn_df = hmr_df.merge(total_itns_df, how='left', on='hhid')

In [60]:
# ITN capacity
itn_df['itn_capacity'] = itn_df['total_itns'] * 2

# Household size 
hh_df = itn_df[['hhid', 'hvidx']]
size = hh_df.groupby('hhid')['hvidx'].nunique().reset_index()
size.columns = ['hhid', 'hh_size']

In [61]:
itn_df = itn_df.merge(size, how='left', on='hhid')

In [62]:
# Compute ratio
itn_df['access_ratio'] = np.minimum(itn_df['itn_capacity'], itn_df['hh_size']) / itn_df['hh_size']

In [63]:
# Calculate access
itn_df['weighted_ratio'] = itn_df.hv005 * itn_df.access_ratio

# Group by region
region_df_access = (itn_df[['hv024', 'hv005', 'weighted_ratio']]
    .groupby('hv024', observed=True)
    .sum()
    .reset_index()
)

# Drop refugee settlements
region_df_access = region_df_access[region_df_access['hv005'] != 0]

# Divide
region_df_access['access_rate'] = region_df_access['weighted_ratio'] / region_df_access['hv005']


In [64]:
region_df_access

,hv024,hv005,weighted_ratio,access_rate
0,kampala,1014956700,6.485719e+08,0.639014
1,south buganda,7169112974,4.902762e+09,0.683873
2,north buganda,5120003785,3.701381e+09,0.722925
3,busoga,3797009509,2.580566e+09,0.679631
4,bukedi,2585607226,1.800876e+09,0.696500
5,elgon,2059554498,1.572741e+09,0.763632
6,teso,2367811911,1.885349e+09,0.796241
7,lango,2437319984,1.905078e+09,0.781628
8,acholi,1953488373,1.174901e+09,0.601437
9,karamoja,1424291664,8.264528e+08,0.580255


# Reconciliation

In [ ]:
region_df_use = region_df_use.drop('hv005', axis=1)

In [ ]:
final_df = region_df_use.merge(region_df_access, how='left', on='hv024')

In [79]:
final_df = final_df.rename(columns={'hv024': 'region', 'hv005': 'weight'})

In [86]:
mis = pd.DataFrame({'region': 
    ['kampala', 'south buganda', 'north buganda',
    'busoga', 'bukedi', 'elgon', 'teso', 'lango', 'acholi', 'karamoja', 'west Nile',
    'bunyoro', 'toro', 'ankole', 'kigezi'], 
    "mis_access": [64, 68, 72, 68, 70, 76, 80, 78, 60, 58, 72, 76, 68, 77, 78], 
    "mis_use": [55, 59, 66, 65, 65, 68, 78, 77, 58, 55, 70, 70, 55, 72, 68]})




In [88]:
comparison = mis.merge(final_df, how='left', on='region')

In [93]:
comparison = comparison.drop(['slept_ITN', 'use_numerator', 'weight', 'weighted_ratio'], axis=1)

In [97]:
styled_comp = (comparison.style
                .format(precision=3))

styled_comp

,region,mis_access,mis_use,use_rate,access_rate
0,kampala,64,55,0.552,0.639
1,south buganda,68,59,0.592,0.684
2,north buganda,72,66,0.657,0.723
3,busoga,68,65,0.648,0.680
4,bukedi,70,65,0.650,0.697
5,elgon,76,68,0.675,0.764
6,teso,80,78,0.779,0.796
7,lango,78,77,0.768,0.782
8,acholi,60,58,0.583,0.601
9,karamoja,58,55,0.549,0.580


## Export

In [73]:
path = Path("../data/processed")
path.mkdir(exist_ok=True)
final_df.to_csv("../data/processed/access_use_rates.csv", index=False)
